# Hostile workflows: three analysis arms on six traps

The v4 battery gives each analyst a dataset with one planted trap. The traps are
missing values, a leaked target, heteroscedastic noise, confounding, a steep but
smooth transition that looks like a jump, and a real diabetes dataset with a sealed
test set. Each arm analyses each case alone and reports a fitted model and its
conclusions. The aim is to see whether the arms find the planted structure, avoid the
trap, and state which conclusions the data support.

Three arms take part:

- **A**: mboostrs, with the mboostrs agent skill.
- **B**: scikit-learn and XGBoost, with no mboostrs.
- **C**: mboostrs, without the skill.

Each arm ran each of the six cases, so there are 18 runs. The protocol is
`docs/benchmarks/agentic_hostile_workflows_protocol.md`, and the graded report is
`docs/benchmarks/agentic_hostile_workflows_report.md`.

## Grades and numbers

The report grades each run on a local v4 rubric. Those grades are historical. They are
not scores under the current scorecard, and this page does not restate them. It shows
the numbers the grades rest on: the planted effects, the cost of each run, and the
held-out error on the real case.

## Planted effects and arm estimates

Each case has a planted effect that a correct analysis should recover. The table lists
the truth from `TRUTH_KEY.json` and each arm's estimate from the report. Bold marks the
estimate closest to the truth in each row. The sine amplitude in the missingness case
and the variance structure in the heteroscedastic case are described in the text,
because the report gives them in different units.

In [1]:
import json
from pathlib import Path

import polars as pl
from great_tables import GT

# This page runs from docs/book/benchmarks; the planted truth sits with the battery.
TRUTH = Path("evidence/hostile_workflows/TRUTH_KEY.json")
truth_key = json.loads(TRUTH.read_text())
print("cases in the truth key:", [k for k in truth_key if isinstance(truth_key[k], dict)])

cases in the truth key: ['missingness', 'leakage', 'heteroscedastic', 'confounding', 'sharp_smooth', 'real']


In [2]:
# Estimates as reported in the graded report, per arm. Each row names one planted quantity.
recovery = [
    {"Case": "Missingness", "Quantity": "slope above the kink at x0 = 0.40", "Truth": 2.5, "A": 2.4, "B": 2.4, "C": 2.41},
    {"Case": "Missingness", "Quantity": "linear effect of x3", "Truth": 1.0, "A": 0.98, "B": 1.05, "C": 1.07},
    {"Case": "Leakage", "Quantity": "effect of x0 after excluding the leaked column", "Truth": 2.0, "A": 2.10, "B": 2.08, "C": 2.0},
    {"Case": "Confounding", "Quantity": "within-group effect of x0", "Truth": 1.5, "A": 1.58, "B": 1.58, "C": 1.58},
    {"Case": "Confounding", "Quantity": "group effect of g", "Truth": -2.0, "A": -2.04, "B": -2.04, "C": -2.04},
]

rows = []
for r in recovery:
    arm_values = {"A": r["A"], "B": r["B"], "C": r["C"]}
    closest = min(arm_values, key=lambda arm: abs(arm_values[arm] - r["Truth"]))
    entry = {"Case": r["Case"], "Quantity": r["Quantity"], "Truth": r["Truth"]}
    for arm in ["A", "B", "C"]:
        text = f"{arm_values[arm]:+.2f}" if r["Truth"] < 0 else f"{arm_values[arm]:.2f}"
        entry[f"Arm {arm}"] = f"**{text}**" if arm == closest else text
    rows.append(entry)

recovery_table = pl.DataFrame(rows)
(
    GT(recovery_table)
    .tab_header(
        title="Planted effects and each arm's estimate",
        subtitle="Estimate against the planted truth; bold marks the arm closest to the truth in each row",
    )
    .fmt_markdown(columns=["Arm A", "Arm B", "Arm C"])
    .fmt_number(columns=["Truth"], decimals=2)
    .tab_stubhead(label="Case")
    .tab_source_note(
        "Arm A: mboostrs with the skill; arm B: scikit-learn and XGBoost; arm C: mboostrs without the skill. "
        "Truth from TRUTH_KEY.json; estimates from the graded report."
    )
)

GT(_tbl_data=shape: (5, 6)
┌─────────────┬─────────────────────────────────┬───────┬───────────┬───────┬──────────┐
│ Case        ┆ Quantity                        ┆ Truth ┆ Arm A     ┆ Arm B ┆ Arm C    │
│ ---         ┆ ---                             ┆ ---   ┆ ---       ┆ ---   ┆ ---      │
│ str         ┆ str                             ┆ f64   ┆ str       ┆ str   ┆ str      │
╞═════════════╪═════════════════════════════════╪═══════╪═══════════╪═══════╪══════════╡
│ Missingness ┆ slope above the kink at x0 = 0… ┆ 2.5   ┆ 2.40      ┆ 2.40  ┆ **2.41** │
│ Missingness ┆ linear effect of x3             ┆ 1.0   ┆ **0.98**  ┆ 1.05  ┆ 1.07     │
│ Leakage     ┆ effect of x0 after excluding t… ┆ 2.0   ┆ 2.10      ┆ 2.08  ┆ **2.00** │
│ Confounding ┆ within-group effect of x0       ┆ 1.5   ┆ **1.58**  ┆ 1.58  ┆ 1.58     │
│ Confounding ┆ group effect of g               ┆ -2.0  ┆ **-2.04** ┆ -2.04 ┆ -2.04    │
└─────────────┴─────────────────────────────────┴───────┴───────────┴───────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x10ad6f6b0>, _boxhead=Boxhead([ColInfo(var='Case', type=<ColInfoTypeEnum.default: 1>, column_label='Case', column_align='left', column_width=None), ColInfo(var='Quantity', type=<ColInfoTypeEnum.default: 1>, column_label='Quantity', column_align='left', column_width=None), ColInfo(var='Truth', type=<ColInfoTypeEnum.default: 1>, column_label='Truth', column_align='right', column_width=None), ColInfo(var='Arm A', type=<ColInfoTypeEnum.default: 1>, column_label='Arm A', column_align='left', column_width=None), ColInfo(var='Arm B', type=<ColInfoTypeEnum.default: 1>, column_label='Arm B', column_align='left', column_width=None), ColInfo(var='Arm C', type=<ColInfoTypeEnum.default: 1>, column_label='Arm C', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10a680980>, _spanners=Spanners([]), _heading=Heading(title="Planted effects and each arm's estimate", subtitle='Estimate against the planted truth; bold marks the arm closest to the truth in each row', preheader=None), _stubhead='Case', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10a47da60>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10a47c140>, _source_notes=['Arm A: mboostrs with the skill; arm B: scikit-learn and XGBoost; arm C: mboostrs without the skill. Truth from TRUTH_KEY.json; estimates from the graded report.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10a053080>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10abd72f0>, <great_tables._gt_data.FormatInfo object at 0x10adaa2a0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_ligh

All three arms recover the planted effects to within about 0.1 of the truth on these
rows, and the differences between arms are smaller than the spread of the estimates.
The report states that every arm got the core verdict right on all six cases. The
heteroscedastic case has no single planted number, because the noise is planted as a
standard deviation of $0.25 + 1.5 x_2$. The arms estimated a noise ratio of about 5 to
1 across $x_2$. The report does not put the true ratio on the same scale, so the page
does not compare the two.

The sharp-smooth case has no true jump, and every arm correctly typed the
transition as smooth.

## Cost of each run

The efficiency table gives the wall-clock time in minutes and the fit count for each
run. Fit counts are self-reported, and the agents counted them by different conventions,
so the fit totals are a rough guide. The totals below are summed from the table, and
they match the report's totals.

In [3]:
# Wall-clock minutes and self-reported fit counts, as written in the graded report.
cases = ["Missingness", "Leakage", "Heteroscedastic", "Confounding", "Sharp smooth", "Real"]
minutes = {"A": [23, 19, 24, 16, 20, 25], "B": [12, 8, 13, 3.5, 12, 9.5], "C": [10, 11, 18, 19, 20, 9]}
fits = {"A": [60, 150, 50, 16, 150, 260], "B": [11, 60, 24, 45, 60, 155], "C": [42, 31, 60, 110, 75, 60]}

cost_rows = []
for i, case in enumerate(cases):
    entry = {"Case": case}
    for arm in ["A", "B", "C"]:
        entry[f"{arm} minutes"] = minutes[arm][i]
        entry[f"{arm} fits"] = fits[arm][i]
    cost_rows.append(entry)

total = {"Case": "Total"}
for arm in ["A", "B", "C"]:
    total[f"{arm} minutes"] = sum(minutes[arm])
    total[f"{arm} fits"] = sum(fits[arm])
cost_rows.append(total)

# The report's own totals, for a check on the transcription.
assert total["A minutes"] == 127 and total["B minutes"] == 58 and total["C minutes"] == 87
assert total["A fits"] == 686 and total["B fits"] == 355 and total["C fits"] == 378

cost_table = pl.DataFrame(cost_rows)
(
    GT(cost_table)
    .tab_header(
        title="Wall-clock time and reported fits per run",
        subtitle="Minutes of wall-clock time and self-reported fit counts; lower is cheaper",
    )
    .tab_spanner(label="Arm A (mboostrs with skill)", columns=["A minutes", "A fits"])
    .tab_spanner(label="Arm B (scikit-learn and XGBoost)", columns=["B minutes", "B fits"])
    .tab_spanner(label="Arm C (mboostrs without skill)", columns=["C minutes", "C fits"])
    .cols_label(**{"A minutes": "Minutes", "A fits": "Fits", "B minutes": "Minutes", "B fits": "Fits", "C minutes": "Minutes", "C fits": "Fits"})
    .tab_stubhead(label="Case")
    .tab_source_note(
        "Time includes agent deliberation, not only compute. Fit counts follow each agent's own convention. "
        "Source: efficiency table in agentic_hostile_workflows_report.md."
    )
)

GT(_tbl_data=shape: (7, 7)
┌─────────────────┬───────────┬────────┬───────────┬────────┬───────────┬────────┐
│ Case            ┆ A minutes ┆ A fits ┆ B minutes ┆ B fits ┆ C minutes ┆ C fits │
│ ---             ┆ ---       ┆ ---    ┆ ---       ┆ ---    ┆ ---       ┆ ---    │
│ str             ┆ i64       ┆ i64    ┆ f64       ┆ i64    ┆ i64       ┆ i64    │
╞═════════════════╪═══════════╪════════╪═══════════╪════════╪═══════════╪════════╡
│ Missingness     ┆ 23        ┆ 60     ┆ 12.0      ┆ 11     ┆ 10        ┆ 42     │
│ Leakage         ┆ 19        ┆ 150    ┆ 8.0       ┆ 60     ┆ 11        ┆ 31     │
│ Heteroscedastic ┆ 24        ┆ 50     ┆ 13.0      ┆ 24     ┆ 18        ┆ 60     │
│ Confounding     ┆ 16        ┆ 16     ┆ 3.5       ┆ 45     ┆ 19        ┆ 110    │
│ Sharp smooth    ┆ 20        ┆ 150    ┆ 12.0      ┆ 60     ┆ 20        ┆ 75     │
│ Real            ┆ 25        ┆ 260    ┆ 9.5       ┆ 155    ┆ 9         ┆ 60     │
│ Total           ┆ 127       ┆ 686    ┆ 58.0      ┆ 355    ┆ 87        ┆ 378    │
└─────────────────┴───────────┴────────┴───────────┴────────┴───────────┴────────┘, _body=<great_tables._gt_data.Body object at 0x10a075820>, _boxhead=Boxhead([ColInfo(var='Case', type=<ColInfoTypeEnum.default: 1>, column_label='Case', column_align='left', column_width=None), ColInfo(var='A minutes', type=<ColInfoTypeEnum.default: 1>, column_label='Minutes', column_align='right', column_width=None), ColInfo(var='A fits', type=<ColInfoTypeEnum.default: 1>, column_label='Fits', column_align='right', column_width=None), ColInfo(var='B minutes', type=<ColInfoTypeEnum.default: 1>, column_label='Minutes', column_align='right', column_width=None), ColInfo(var='B fits', type=<ColInfoTypeEnum.default: 1>, column_label='Fits', column_align='right', column_width=None), ColInfo(var='C minutes', type=<ColInfoTypeEnum.default: 1>, column_label='Minutes', column_align='right', column_width=None), ColInfo(var='C fits', type=<ColInfoTypeEnum.default: 1>, column_label='Fits', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10b7581a0>, _spanners=Spanners([SpannerInfo(spanner_id='Arm A (mboostrs with skill)', spanner_level=0, spanner_label='Arm A (mboostrs with skill)', spanner_units=None, spanner_pattern=None, vars=['A minutes', 'A fits'], built=None), SpannerInfo(spanner_id='Arm B (scikit-learn and XGBoost)', spanner_level=0, spanner_label='Arm B (scikit-learn and XGBoost)', spanner_units=None, spanner_pattern=None, vars=['B minutes', 'B fits'], built=None), SpannerInfo(spanner_id='Arm C (mboostrs without skill)', spanner_level=0, spanner_label='Arm C (mboostrs without skill)', spanner_units=None, spanner_pattern=None, vars=['C minutes', 'C fits'], built=None)]), _heading=Heading(title='Wall-clock time and reported fits per run', subtitle='Minutes of wall-clock time and self-reported fit counts; lower is cheaper', preheader=None), _stubhead='Case', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10a682ff0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10a6837d0>, _source_notes=["Time includes agent deliberation, not only compute. Fit counts follow each agent's own convention. Source: efficiency table in agentic_hostile_workflows_report.md."], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10b745040>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='val

Arm A took the most time in four cases: missingness, leakage, heteroscedastic and real.
On sharp smooth it took the same 20 minutes as arm C. On confounding it took 16
minutes, the least of the three arms with C at 19. The report says the skill arm was
slowest on five of six cases. The table shows four clear cases, one tie and one case
where A was not the slowest, so the report's count is one case higher than the numbers
support.

Arm A also made the most fits in total: 686, against 378 for C and 355 for B. Its
largest counts are on leakage, 150 fits, and on the real case, 260 fits.

## Real data

The real case is the scikit-learn diabetes dataset, split 80/20 with a fixed seed. The
test rows are sealed, so the error below is the held-out error of each arm's final
model. The cross-validation estimate is the arm's own estimate from training rows.

In [4]:
real = pl.DataFrame(
    [
        {"Arm": "A (mboostrs with skill)", "Held-out RMSE": 58.51, "Cross-validation RMSE": 53.5, "CV spread": "not reported"},
        {"Arm": "B (scikit-learn and XGBoost)", "Held-out RMSE": 59.74, "Cross-validation RMSE": 53.99, "CV spread": "± 2.69"},
        {"Arm": "C (mboostrs without skill)", "Held-out RMSE": 59.13, "Cross-validation RMSE": 54.3, "CV spread": "± 0.4"},
    ]
)
best_rmse = real["Held-out RMSE"].min()
real = real.with_columns(
    pl.when(pl.col("Held-out RMSE") == best_rmse)
    .then(pl.format("**{}**", pl.col("Held-out RMSE").cast(pl.String)))
    .otherwise(pl.col("Held-out RMSE").cast(pl.String))
    .alias("Held-out RMSE")
)
(
    GT(real)
    .tab_header(
        title="Held-out RMSE on the sealed diabetes test set",
        subtitle="Disease progression, 88 test rows; lower is better; bold marks the lowest held-out error",
    )
    .fmt_markdown(columns=["Held-out RMSE"])
    .tab_stubhead(label="Arm")
    .tab_source_note(
        "Test rows are sealed and not in the repository. Cross-validation spread as reported. The report calls the "
        "differences of about 1 to 2 points a tie at this sample size."
    )
)

GT(_tbl_data=shape: (3, 4)
┌──────────────────────────────┬───────────────┬───────────────────────┬──────────────┐
│ Arm                          ┆ Held-out RMSE ┆ Cross-validation RMSE ┆ CV spread    │
│ ---                          ┆ ---           ┆ ---                   ┆ ---          │
│ str                          ┆ str           ┆ f64                   ┆ str          │
╞══════════════════════════════╪═══════════════╪═══════════════════════╪══════════════╡
│ A (mboostrs with skill)      ┆ **58.51**     ┆ 53.5                  ┆ not reported │
│ B (scikit-learn and XGBoost) ┆ 59.74         ┆ 53.99                 ┆ ± 2.69       │
│ C (mboostrs without skill)   ┆ 59.13         ┆ 54.3                  ┆ ± 0.4        │
└──────────────────────────────┴───────────────┴───────────────────────┴──────────────┘, _body=<great_tables._gt_data.Body object at 0x10b759040>, _boxhead=Boxhead([ColInfo(var='Arm', type=<ColInfoTypeEnum.default: 1>, column_label='Arm', column_align='left', column_width=None), ColInfo(var='Held-out RMSE', type=<ColInfoTypeEnum.default: 1>, column_label='Held-out RMSE', column_align='left', column_width=None), ColInfo(var='Cross-validation RMSE', type=<ColInfoTypeEnum.default: 1>, column_label='Cross-validation RMSE', column_align='right', column_width=None), ColInfo(var='CV spread', type=<ColInfoTypeEnum.default: 1>, column_label='CV spread', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10a683470>, _spanners=Spanners([]), _heading=Heading(title='Held-out RMSE on the sealed diabetes test set', subtitle='Disease progression, 88 test rows; lower is better; bold marks the lowest held-out error', preheader=None), _stubhead='Arm', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10b759370>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10b759220>, _source_notes=['Test rows are sealed and not in the repository. Cross-validation spread as reported. The report calls the differences of about 1 to 2 points a tie at this sample size.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10b759160>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10af9aba0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type

The three arms are within 1.2 points of each other on the sealed test set, and the
report calls this a tie. With 88 test rows the difference is within sampling noise. The
gap between each cross-validation estimate and the held-out error is 4 to 6 points, and
the report attributes that gap to the size of the test set.

## Findings

- **The three arms reach the same conclusions on the planted effects.** Each arm
  recovers the planted slopes to within about 0.1, and each reports the Simpson-style
  reversal in the confounding case without overclaiming causation.
- **Leakage detection.** Each arm identified and excluded the column that equals
  the target plus small noise.
- **On the real case, the arms are tied.** The spread of held-out error is 1.2 points
  on 88 rows.
- **Arm A is the most expensive in time and in fits.** It used 127 minutes and 686 fits
  over the six cases, against 58 minutes and 355 fits for arm B, and 87 minutes and 378
  fits for arm C. The report notes that the difference does not come with a difference in
  correctness.
- **Arm B matched mboostrs on every planted case.** It detected the heteroscedasticity
  that scikit-learn does not model directly, and it reported the limit of its
  point-prediction methods for that case.

## Limits

- **Historical grading.** The v4 local rubric is archived. Its grades are not scores under
  the current scorecard, and the report says so.
- **Self-reported fits and wall-clock.** The fit counts follow each agent's convention,
  and wall-clock includes deliberation. Read the totals as rough guides.
- **Pre-fix discontinuity screen.** The agents ran before a fix to the discontinuity
  screen. The sharp-smooth verdicts do not extend to the steep transitions that the fix
  addresses. The battery's transition has slope 4.5 per unit, and a steeper case is
  needed to test the fix.
- **One reference tree search hung.** On the heteroscedastic data, the reference tree
  step of the library hung twice, and arm A skipped that step and reported the skip.
  The report files this as a library issue.
- **Real-case noise.** At 88 test rows, differences of about one point do not separate
  the arms.